# Imports

In [1]:
import sklearn as sk
import numpy as np

In [2]:
import sys
import os
sys.path.insert(0, os.path.abspath('..'))

In [3]:
np.set_printoptions(linewidth=100, precision=2)

In [4]:
%load_ext autoreload
%autoreload 2

# Iris

In [5]:
from utils.utils import load_iris, split_db_2to1

D,L = load_iris()
(DTR, LTR), (DTE, LTE) = split_db_2to1(D, L)

## Gaussian Models

### MVG

In [6]:
from utils.utils import vcol, get_cov

u0 = vcol(DTR[:, LTR==0].mean(1))
c0 = get_cov(DTR[:, LTR==0])


u1 = vcol(DTR[:, LTR==1].mean(1))
c1 = get_cov(DTR[:, LTR==1])

u2 = vcol(DTR[:, LTR==2].mean(1))
c2 = get_cov(DTR[:, LTR==2])

In [7]:
from utils.utils import loglikelihood, logpdf_GAU_ND

logpdf0 = logpdf_GAU_ND(DTE, u0, c0)
logpdf1 = logpdf_GAU_ND(DTE, u1, c1)
logpdf2 = logpdf_GAU_ND(DTE, u2, c2)

In [8]:
logS = np.vstack((logpdf0, logpdf1, logpdf2))

In [9]:
Pc = np.ones((3,1))/3

### Working Directly with densities

In [10]:
S = np.exp(logS)

In [11]:
SJoint = S*Pc

In [12]:
SJoint_MVG = np.load('../data/SJoint_MVG.npy')
print(np.sum(SJoint-SJoint_MVG))

1.785565223336073e-14


In [13]:
from utils.utils import vrow

SMarginal = vrow(SJoint.sum(0))

In [14]:
SPost = SJoint / SMarginal

In [15]:
pred = np.argmax(SPost, axis=0)

In [16]:
acc = np.sum(LTE==pred) / len(LTE)
err = 1 - acc
print(err)

0.040000000000000036


### Working in Log Domain

In [17]:
logPc = np.log(Pc)

In [18]:
logSJoint = logS + logPc

In [19]:
from scipy.special import logsumexp

logSMarginal = vrow(logsumexp(logSJoint, axis=0))


In [20]:
logSPost = logSJoint - logSMarginal
SPost = np.exp(logSPost)


In [21]:
pred = np.argmax(SPost, axis=0)

In [22]:
acc = np.sum(LTE==pred) / len(LTE)
err = 1 - acc
print(err)

0.040000000000000036


### Naive Bayes

In [23]:
logpdf0 = logpdf_GAU_ND(DTE, u0, c0 * np.eye(c0.shape[0], c0.shape[1]))
logpdf1 = logpdf_GAU_ND(DTE, u1, c1 * np.eye(c1.shape[0], c1.shape[1]))
logpdf2 = logpdf_GAU_ND(DTE, u2, c2 * np.eye(c2.shape[0], c2.shape[1]))

In [24]:
logS = np.vstack((logpdf0, logpdf1, logpdf2))
logSJoint = logS + logPc
logSMarginal = vrow(logsumexp(logSJoint, axis=0))
logSPost = logSJoint - logSMarginal
SPost = np.exp(logSPost)

In [25]:
pred = np.argmax(SPost, axis=0)

In [26]:
acc = np.sum(LTE==pred) / len(LTE)
err = 1 - acc
print(err)

0.040000000000000036


### Tied Variance

In [27]:
from utils.utils import get_class_covariances

_, Sw = get_class_covariances(DTR, LTR, [0, 1, 2])

In [28]:
logpdf0 = logpdf_GAU_ND(DTE, u0, Sw)
logpdf1 = logpdf_GAU_ND(DTE, u1, Sw)
logpdf2 = logpdf_GAU_ND(DTE, u2, Sw)

In [29]:
logS = np.vstack((logpdf0, logpdf1, logpdf2))
logSJoint = logS + logPc
logSMarginal = vrow(logsumexp(logSJoint, axis=0))
logSPost = logSJoint - logSMarginal
SPost = np.exp(logSPost)

In [30]:
pred = np.argmax(SPost, axis=0)

In [31]:
acc = np.sum(LTE==pred) / len(LTE)
err = 1 - acc
print(err)

0.020000000000000018


## Binary Tasks

In [32]:
D = D[:, L != 0]
L = L[L != 0]
(DTR, LTR), (DTE, LTE) = split_db_2to1(D, L)
labels = [1, 2]

In [33]:
u1 = vcol(DTR[:, LTR==1].mean(1))
c1 = get_cov(DTR[:, LTR==1])

u2 = vcol(DTR[:, LTR==2].mean(1))
c2 = get_cov(DTR[:, LTR==2])

In [34]:
logpdf1 = logpdf_GAU_ND(DTE, u1, c1)
logpdf2 = logpdf_GAU_ND(DTE, u2, c2)

In [35]:
llr = logpdf2 - logpdf1

In [36]:
P1 = P2 = 0.5
t = 0

In [37]:
pred = np.where(llr >= t, 2, 1)

In [38]:
acc = np.sum(LTE==pred) / len(LTE)
err = 1 - acc
print(err)

0.08823529411764708


## Testing my own classifier Class

In [39]:
from utils.utils import load_iris, split_db_2to1

D,L = load_iris()
(DTR, LTR), (DTE, LTE) = split_db_2to1(D, L)

### Multiclass

In [40]:
from utils.BayesClassifier import MultiClassMVG, MultiClassNaiveBayes, MultiClassTiedVariance

mvg = MultiClassMVG()
nb = MultiClassNaiveBayes()
tv = MultiClassTiedVariance()

In [41]:
Pc = np.ones((3,1))/3

In [42]:
mvg.fit(DTR, LTR)
nb.fit(DTR, LTR)
tv.fit(DTR, LTR)

mvg.set_priors(Pc)
nb.set_priors(Pc)
tv.set_priors(Pc)

In [43]:
mvg.predict(DTE)
nb.predict(DTE)
tv.predict(DTE)

In [45]:
print("MVG: ", f"{mvg.evaluate(LTE)[0]:.4f}")
print("Naive Bayes: ", f"{nb.evaluate(LTE)[0]:.4f}")
print("Tied Variance: ", f"{tv.evaluate(LTE)[0]:.4f}")

MVG:  0.0400
Naive Bayes:  0.0400
Tied Variance:  0.0200


### Binary

In [ ]:
D = D[:, L != 0]
L = L[L != 0]
L[L == 1] = 0
L[L == 2] = 1
(DTR, LTR), (DTE, LTE) = split_db_2to1(D, L)

In [ ]:
from utils.BayesClassifier import BinaryMVG

mvg = BinaryMVG()
Pc = np.ones((2,1))/2
mvg.set_priors(np.ones((2,1))/2)
mvg.set_threshold_via_prior_ratio()
mvg.fit(DTR, LTR)
mvg.predict(DTE)
print("MVG: ", f"{mvg.evaluate(LTE)[0]:.4f}")

# Project

Last lab, we fit a gaussian to each individual feature. It seems features 4 and 5 are not gaussian, so lets analyze the performance without them.

Besides, in 1st lab we saw that features 0|1 have similar mean and different variance, while 2|3 have similar variance but different mean. Lets analyze the performance using only a pair.

Finally, lets try applying PCS  to the data and see the results

| | MVG | Tied | NB |
| -----| --- | ---- | --- |
| All Features | 7% | 9.3% | 7.2% |
| 0, 1, 2, 3 | 7.4% | 9.4% | 7.5% |
| 0, 1 | 36% | 50% | 36% |
| 2, 3 | 9.5% | 9.5% | 9.5% |
| PCA | 8.9% | 9.3% | 9% |

## Data

In [63]:
from utils.utils import load_data

PROJECT_FILE = "../data/trainData.txt"
D, L, labels = load_data(PROJECT_FILE)
feature_names = [f'F{i}' for i in range(0, D.shape[0])]
L = L.astype(int)
labels = [0, 1]
(DTR, LTR), (DTE, LTE) = split_db_2to1(D, L)

## Classifiers, Priors and Thresholds

In [60]:
from utils.BayesClassifier import BinaryMVG, BinaryNaiveBayes, BinaryTiedVariance

mvg = BinaryMVG()
nb = BinaryNaiveBayes()
tv = BinaryTiedVariance()

In [70]:
Pc = np.ones((2,1))/2
mvg.set_priors(Pc)
nb.set_priors(Pc)
tv.set_priors(Pc)

costs = np.array([
    [0, 1],
    [1, 0]
])

mvg.set_cost_matrix(costs)
nb.set_cost_matrix(costs)
tv.set_cost_matrix(costs)

mvg.set_threshold_via_prior_ratio()
nb.set_threshold_via_prior_ratio()
tv.set_threshold_via_prior_ratio()

## All Features

In [64]:
mvg.fit(DTR, LTR)
nb.fit(DTR, LTR)
tv.fit(DTR, LTR)

In [65]:
mvg.predict(DTE)
nb.predict(DTE)
tv.predict(DTE)

In [71]:
print("MVG: ", f"{mvg.evaluate(LTE)[0]:.4f}")
print("Naive Bayes: ", f"{nb.evaluate(LTE)[0]:.4f}")
print("Tied Variance: ", f"{tv.evaluate(LTE)[0]:.4f}")

MVG:  0.0700
Naive Bayes:  0.0720
Tied Variance:  0.0930


### Investigating Feature Independence

Investigating the features, we found low correlation between features, making their independence assumption likely true. That allows the Naive Bayes Classifier to work properly

In [67]:
c0 = get_cov(DTR[:, LTR==0])
c1 = get_cov(DTR[:, LTR==1])

In [68]:
print(c0, "\n\n", c1)

[[ 6.01e-01  5.16e-05  1.91e-02  1.93e-02  1.28e-02 -1.35e-02]
 [ 5.16e-05  1.45e+00 -1.61e-02 -1.59e-02 -2.65e-02  2.29e-02]
 [ 1.91e-02 -1.61e-02  5.65e-01 -1.84e-03 -6.91e-03  1.69e-02]
 [ 1.93e-02 -1.59e-02 -1.84e-03  5.42e-01  5.25e-03  1.36e-02]
 [ 1.28e-02 -2.65e-02 -6.91e-03  5.25e-03  6.96e-01  1.58e-02]
 [-1.35e-02  2.29e-02  1.69e-02  1.36e-02  1.58e-02  6.87e-01]] 

 [[ 1.45e+00 -1.47e-02  5.57e-03  1.57e-02  1.95e-02 -1.77e-04]
 [-1.47e-02  5.53e-01 -1.12e-02 -9.06e-03 -1.47e-02  1.63e-02]
 [ 5.57e-03 -1.12e-02  5.57e-01  2.76e-02 -3.77e-03 -1.46e-02]
 [ 1.57e-02 -9.06e-03  2.76e-02  5.70e-01 -1.17e-02  3.50e-02]
 [ 1.95e-02 -1.47e-02 -3.77e-03 -1.17e-02  1.34e+00  1.69e-02]
 [-1.77e-04  1.63e-02 -1.46e-02  3.50e-02  1.69e-02  1.30e+00]]


In [69]:
Corr0 = c0 / ( vcol(c0.diagonal()**0.5) * vrow(c0.diagonal()**0.5) )
Corr1 = c1 / ( vcol(c1.diagonal()**0.5) * vrow(c1.diagonal()**0.5) )

print(Corr0, "\n\n", Corr1)

[[ 1.00e+00  5.53e-05  3.27e-02  3.37e-02  1.98e-02 -2.10e-02]
 [ 5.53e-05  1.00e+00 -1.78e-02 -1.79e-02 -2.64e-02  2.30e-02]
 [ 3.27e-02 -1.78e-02  1.00e+00 -3.33e-03 -1.10e-02  2.71e-02]
 [ 3.37e-02 -1.79e-02 -3.33e-03  1.00e+00  8.55e-03  2.23e-02]
 [ 1.98e-02 -2.64e-02 -1.10e-02  8.55e-03  1.00e+00  2.29e-02]
 [-2.10e-02  2.30e-02  2.71e-02  2.23e-02  2.29e-02  1.00e+00]] 

 [[ 1.00e+00 -1.64e-02  6.20e-03  1.73e-02  1.40e-02 -1.29e-04]
 [-1.64e-02  1.00e+00 -2.02e-02 -1.61e-02 -1.70e-02  1.92e-02]
 [ 6.20e-03 -2.02e-02  1.00e+00  4.89e-02 -4.36e-03 -1.71e-02]
 [ 1.73e-02 -1.61e-02  4.89e-02  1.00e+00 -1.34e-02  4.06e-02]
 [ 1.40e-02 -1.70e-02 -4.36e-03 -1.34e-02  1.00e+00  1.28e-02]
 [-1.29e-04  1.92e-02 -1.71e-02  4.06e-02  1.28e-02  1.00e+00]]


## Selecting Features

In [84]:
D, L, labels = load_data(PROJECT_FILE)
D = D[4:6, :]
L = L.astype(int)

(DTR, LTR), (DTE, LTE) = split_db_2to1(D, L)

In [85]:
mvg.fit(DTR, LTR)
nb.fit(DTR, LTR)
tv.fit(DTR, LTR)


In [86]:
mvg.predict(DTE)
nb.predict(DTE)
tv.predict(DTE)

In [87]:
print("MVG: ", f"{mvg.evaluate(LTE)[0]:.4f}")
print("Naive Bayes: ", f"{nb.evaluate(LTE)[0]:.4f}")
print("Tied Variance: ", f"{tv.evaluate(LTE)[0]:.4f}")

MVG:  0.2710
Naive Bayes:  0.2620
Tied Variance:  0.4895


## Including PCA

In [ ]:
from utils.utils import get_PCs

D, L, labels = load_data(PROJECT_FILE)
L = L.astype(int)
(DTR, LTR), (DTE, LTE) = split_db_2to1(D, L)

P = get_PCs(DTR, 3)
DTR_PCA = P.T @ DTR
DTE_PCA = P.T @ DTE

In [ ]:
mvg.fit(DTR_PCA, LTR)
nb.fit(DTR_PCA, LTR)
tv.fit(DTR_PCA, LTR)

In [ ]:
mvg.predict(DTE_PCA)
nb.predict(DTE_PCA)
tv.predict(DTE_PCA)

In [ ]:

print("MVG: ", f"{mvg.evaluate(LTE):.4f}")
print("Naive Bayes: ", f"{nb.evaluate(LTE):.4f}")
print("Tied Variance: ", f"{tv.evaluate(LTE):.4f}")